#### Using python [sqlframe](https://github.com/eakmanrq/sqlframe) library to use PySpark dataframe API against a PostgreSQL database

In [1]:
from dotenv import load_dotenv
import os
from psycopg2 import connect
from sqlframe.postgres import functions as F
from sqlframe.postgres import PostgresSession

In [2]:
load_dotenv()

conn = connect(
    dbname=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    host=os.environ["DB_HOST"],
    port=os.environ["DB_PORT"],
)

"""One caveat: autocommit mode means each statement is its own transaction, so if a multi-statement write ever needs to be atomic
(e.g., you want the drop and create to succeed or fail together), you'd lose that guarantee. For a single saveAsTable() call this isn't a concern,
but keep it in mind if your script grows."""
# conn.autocommit = True  # ensures DDL/writes are visible to other connections immediately

session = PostgresSession(conn=conn)

#### Reading a PostgreSQL table as a PySpark dataframe

In [3]:
df = (
    session.table('public.vehicles')
    .where(F.col("year") == "2027")
    .select("year","make","model","fueltype","fueltype1","fueltype2")
)

sqlframe has features that PySpark does not have like `saveAsTable()` method, which allows you to save a dataframe as a table with just one line of code.  Unfortunately, there is currently a bug where if using PostgreSQL as the backend, the `saveAsTable()` method will fail because the underlying SQL that sqlframe generates is using a dialect using `CREATE OR REPLACE TABLE` syntax which PostgreSQL does not support.

In [ ]:
df.write.mode("overwrite").saveAsTable("my_new_table")
conn.commit()

**Workaround:** Execute with correct SQL statements using connection cursor

In [4]:
# Drop the target table first (if it exists), since sqlframe's
# mode("overwrite") emits "CREATE OR REPLACE TABLE", which Postgres
# doesn't support.
with conn.cursor() as cur:
    cur.execute('DROP TABLE IF EXISTS public.models_2027')
    conn.commit()

# No mode() needed now — the table doesn't exist, so this generates
# a plain CREATE TABLE ... AS SELECT
df.write.saveAsTable('public.models_2027')
conn.commit()

#### Let's check that our new table was actually made and has data in it

In [5]:
table = session.table("public.models_2027")
table.limit(5).show()

+------+----------+-------------------+----------+------------------+-----------+
| year |   make   |       model       | fueltype |    fueltype1     | fueltype2 |
+------+----------+-------------------+----------+------------------+-----------+
| 2027 | Chrysler |    Pacifica AWD   | Regular  | Regular Gasoline |           |
| 2027 |  Lotus   |       Emira       | Regular  | Regular Gasoline |           |
| 2027 |   BMW    |     430i Coupe    | Premium  | Premium Gasoline |           |
| 2027 |   BMW    | 430i xDrive Coupe | Premium  | Premium Gasoline |           |
| 2027 |   BMW    |  430i Convertible | Premium  | Premium Gasoline |           |
+------+----------+-------------------+----------+------------------+-----------+


#### We can also use `listTables()` to obtain a list of tables

In [6]:
tables = session.catalog.listTables()

In [7]:
type(tables)

list

In [8]:
for table in tables:
    print(table)

Table(name='vehicles', catalog='postgres', namespace=['public'], description=None, tableType='MANAGED', isTemporary=False)
Table(name='models_2027', catalog='postgres', namespace=['public'], description=None, tableType='MANAGED', isTemporary=False)
